# Birkeland's Norwegian Aurora Polaris Expedition (1908) — Implementation
# Birkeland의 노르웨이 오로라 극지 탐험 (1908) — 구현

Simulating the key physics that Birkeland investigated:
charged particle motion in magnetic fields, the terrella experiment,
and auroral precipitation patterns.

Birkeland가 연구한 핵심 물리학을 시뮬레이션합니다:
자기장 내 하전 입자 운동, terrella 실험, 오로라 강하 패턴.

**Contents / 목차:**
1. Earth's Dipole Magnetic Field — 지구 쌍극 자기장
2. Charged Particle Motion (Lorentz Force) — 하전 입자 운동
3. Magnetic Mirror Effect — 자기 거울 효과
4. Terrella Experiment Simulation — Terrella 실험 시뮬레이션
5. Auroral Zone Formation — 오로라대 형성
6. Birkeland Currents Visualization — Birkeland 전류 시각화
7. Connection to Modern Space Weather — 현대 우주 기상과의 연결

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp

plt.rcParams.update({
    "figure.figsize": (10, 8),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

# Physical constants
R_E = 6.371e6        # Earth radius [m]
B_0 = 3.12e-5        # Equatorial surface field [T]
q_e = -1.602e-19     # Electron charge [C]
m_e = 9.109e-31      # Electron mass [kg]
q_p = 1.602e-19      # Proton charge [C]
m_p = 1.673e-27      # Proton mass [kg]

## Part 1: Earth's Dipole Magnetic Field / 지구 쌍극 자기장

Earth's field is approximately a magnetic dipole. Birkeland understood that
this geometry concentrates charged particles at the poles — the key to aurora.

$$\vec{B}(r, \lambda) = \frac{B_0 R_E^3}{r^3}\left(2\sin\lambda\;\hat{r} + \cos\lambda\;\hat{\lambda}\right)$$

$$|\vec{B}| = \frac{B_0 R_E^3}{r^3}\sqrt{1 + 3\sin^2\lambda}$$

지구 자기장은 근사적으로 쌍극자입니다. Birkeland는 이 기하학적 구조가
하전 입자를 극지방으로 집중시킨다는 것을 이해했습니다 — 오로라의 핵심입니다.

In [ ]:
def dipole_field_lines():
    """Visualize Earth's dipole magnetic field and auroral zones."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # --- Left: Dipole field lines in meridional plane ---
    ax = axes[0]

    # Draw Earth
    earth = Circle((0, 0), 1.0, color="steelblue", alpha=0.7, zorder=5)
    ax.add_patch(earth)
    ax.text(0, 0, "Earth", ha="center", va="center", fontsize=10,
            color="white", fontweight="bold", zorder=6)

    # Dipole field lines: r = L * cos^2(lambda)
    for L in [2, 3, 4, 5, 7, 10]:
        lam = np.linspace(-np.pi / 2, np.pi / 2, 300)
        r = L * np.cos(lam) ** 2
        x = r * np.cos(lam)
        z = r * np.sin(lam)

        # Only plot where r > 1 (outside Earth)
        mask = r > 1.0
        ax.plot(x[mask], z[mask], "darkorange", lw=1.2, alpha=0.7)
        ax.plot(-x[mask], z[mask], "darkorange", lw=1.2, alpha=0.7)

    # Mark auroral zones (~67° latitude, L~5 field lines)
    for sign in [1, -1]:
        lat_aurora = np.radians(67)
        ax.plot([0.95 * np.cos(sign * lat_aurora)],
                [0.95 * np.sin(sign * lat_aurora)],
                "r*", ms=15, zorder=7)
        label = "Aurora\n(67°N)" if sign > 0 else "Aurora\n(67°S)"
        ax.annotate(label,
                    (1.1 * np.cos(sign * lat_aurora),
                     1.1 * np.sin(sign * lat_aurora)),
                    fontsize=9, color="red", ha="center")

    ax.set_xlim(-8, 8)
    ax.set_ylim(-6, 6)
    ax.set_aspect("equal")
    ax.set_xlabel("Distance ($R_E$)")
    ax.set_ylabel("Distance ($R_E$)")
    ax.set_title("Earth's Dipole Magnetic Field Lines\n지구 쌍극 자기장선")
    ax.axhline(0, color="gray", ls=":", lw=0.5)
    ax.axvline(0, color="gray", ls=":", lw=0.5)

    # --- Right: Field strength vs latitude ---
    ax2 = axes[1]
    lat_deg = np.linspace(0, 90, 200)
    lat_rad = np.radians(lat_deg)

    # Surface field strength
    B_surface = B_0 * np.sqrt(1 + 3 * np.sin(lat_rad) ** 2)

    ax2.plot(lat_deg, B_surface * 1e6, "darkorange", lw=3)
    ax2.axvline(67, color="red", ls="--", lw=2, alpha=0.7, label="Auroral zone (~67°)")
    ax2.fill_between([63, 70], 0, 70, alpha=0.15, color="red")

    ax2.set_xlabel("Geomagnetic latitude (°) / 지자기 위도 (°)")
    ax2.set_ylabel("Surface field strength (μT) / 표면 자기장 세기 (μT)")
    ax2.set_title("$|B| = B_0\\sqrt{1 + 3\\sin^2\\lambda}$\nField is STRONGEST at poles")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xlim(0, 90)

    ax2.annotate("Equator\n적도: 31 μT", (0, 31.2), fontsize=10,
                 xytext=(15, 35), arrowprops=dict(arrowstyle="->"))
    ax2.annotate("Pole\n극: 62 μT", (90, 62.4), fontsize=10,
                 xytext=(70, 55), arrowprops=dict(arrowstyle="->"))

    plt.tight_layout()
    plt.show()

dipole_field_lines()

## Part 2: Charged Particle Motion in a Dipole Field / 쌍극 자기장에서의 하전 입자 운동

Birkeland proposed that "cathode rays" (electrons) from the Sun spiral along
Earth's field lines into the polar atmosphere. We simulate this by numerically
integrating the Lorentz force equation:

$$m\frac{d\vec{v}}{dt} = q(\vec{v} \times \vec{B})$$

The particle follows a **helical path**, spiraling tighter as it approaches
the poles where $B$ is stronger.

Birkeland는 태양에서 온 전자가 자기장선을 따라 나선형으로 극지방 대기에
진입한다고 제안했습니다. Lorentz force를 수치 적분하여 시뮬레이션합니다.

In [ ]:
def dipole_B_cartesian(x, y, z, B0=1.0, RE=1.0):
    """Compute dipole magnetic field in Cartesian coordinates.

    Args:
        x, y, z: Position in units of RE.
        B0: Equatorial surface field strength.
        RE: Earth radius (normalized to 1).

    Returns:
        Bx, By, Bz components.
    """
    r = np.sqrt(x**2 + y**2 + z**2)
    if r < 0.1:
        return 0.0, 0.0, 0.0
    r5 = r**5
    # Dipole aligned with z-axis, moment m = B0 * RE^3
    m = B0 * RE**3
    Bx = 3 * m * x * z / r5
    By = 3 * m * y * z / r5
    Bz = m * (3 * z**2 - r**2) / r5
    return Bx, By, Bz


def simulate_particle(q_over_m, pos0, vel0, t_max, dt=0.005):
    """Simulate charged particle in a dipole field using Lorentz force.

    Args:
        q_over_m: Charge-to-mass ratio (normalized).
        pos0: Initial position [x, y, z] in R_E.
        vel0: Initial velocity [vx, vy, vz] (normalized).
        t_max: Maximum simulation time.
        dt: Time step.

    Returns:
        positions: Array of (N, 3) positions.
    """
    pos = np.array(pos0, dtype=float)
    vel = np.array(vel0, dtype=float)
    trajectory = [pos.copy()]

    for _ in range(int(t_max / dt)):
        r = np.linalg.norm(pos)
        if r < 1.05 or r > 15:  # hit Earth or escaped
            break

        Bx, By, Bz = dipole_B_cartesian(pos[0], pos[1], pos[2])
        B = np.array([Bx, By, Bz])

        # Lorentz force: F = q(v x B), a = (q/m)(v x B)
        accel = q_over_m * np.cross(vel, B)
        vel = vel + accel * dt
        pos = pos + vel * dt
        trajectory.append(pos.copy())

    return np.array(trajectory)


# Simulate a proton spiraling along field lines toward the pole
# Using normalized units: R_E = 1, B0 = 1, appropriate q/m
trajectory = simulate_particle(
    q_over_m=50.0,        # normalized charge-to-mass ratio
    pos0=[5.0, 0.0, 0.1], # start at L=5 near equatorial plane
    vel0=[0.0, 0.3, 0.15], # initial velocity with parallel component
    t_max=200,
    dt=0.01
)

fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection="3d")

# Draw Earth
u = np.linspace(0, 2 * np.pi, 30)
v = np.linspace(0, np.pi, 20)
xs = np.outer(np.cos(u), np.sin(v))
ys = np.outer(np.sin(u), np.sin(v))
zs = np.outer(np.ones(np.size(u)), np.cos(v))
ax.plot_surface(xs, ys, zs, color="steelblue", alpha=0.3)

# Draw trajectory
ax.plot(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2],
        "darkorange", lw=0.8, alpha=0.8)
ax.plot([trajectory[0, 0]], [trajectory[0, 1]], [trajectory[0, 2]],
        "go", ms=8, label="Start / 시작")
ax.plot([trajectory[-1, 0]], [trajectory[-1, 1]], [trajectory[-1, 2]],
        "r*", ms=12, label="End (aurora!) / 끝 (오로라!)")

# Draw a few field lines for reference
for L in [3, 5, 7]:
    lam = np.linspace(-np.pi / 2, np.pi / 2, 200)
    r_fl = L * np.cos(lam) ** 2
    x_fl = r_fl * np.cos(lam)
    z_fl = r_fl * np.sin(lam)
    ax.plot(x_fl, np.zeros_like(x_fl), z_fl, "gray", lw=0.5, alpha=0.4)

ax.set_xlabel("X ($R_E$)")
ax.set_ylabel("Y ($R_E$)")
ax.set_zlabel("Z ($R_E$)")
ax.set_title("Charged Particle Spiraling Along Dipole Field Lines\n"
             "하전 입자의 쌍극 자기장선을 따른 나선 운동")
ax.legend(loc="upper left")
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_zlim(-4, 4)
plt.tight_layout()
plt.show()

print(f"Trajectory: {len(trajectory)} points")
print(f"Start: r = {np.linalg.norm(trajectory[0]):.1f} R_E")
print(f"End:   r = {np.linalg.norm(trajectory[-1]):.1f} R_E")
print("Particle spirals along field lines toward the pole — Birkeland's key insight!")

## Part 3: Magnetic Mirror & Loss Cone / 자기 거울 & 손실 원뿔

Not all particles reach the atmosphere — some are **reflected** back by the
increasing magnetic field (magnetic mirror effect). Only particles within the
**loss cone** reach low enough altitudes to cause aurora.

$$\mu = \frac{mv_\perp^2}{2B} = \text{const} \quad \Rightarrow \quad \sin^2\alpha_0 = \frac{B_0}{B_{\text{mirror}}}$$

모든 입자가 대기에 도달하는 것은 아닙니다 — 자기장이 강해지면서 일부 입자는
**반사**됩니다 (자기 거울 효과). **손실 원뿔(loss cone)** 안의 입자만이 오로라를 일으킵니다.

In [ ]:
def plot_magnetic_mirror():
    """Demonstrate the magnetic mirror effect and loss cone."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- Left: Magnetic mirror concept ---
    ax1 = axes[0]

    # B field strength along field line (simplified)
    s = np.linspace(-5, 5, 500)  # distance along field line
    B = 1 + 2 * np.cosh(s / 2) / np.cosh(0 / 2)  # stronger at ends (poles)
    B = B / B.min()  # normalize

    ax1.fill_between(s, B, alpha=0.15, color="darkorange")
    ax1.plot(s, B, "darkorange", lw=3, label="$|B|$ along field line")

    # Particle trajectories
    # Particle 1: large pitch angle → reflected (mirrored)
    s_mirror = np.linspace(-3, 3, 100)
    ax1.annotate("", xy=(-3, 2.2), xytext=(0, 1.0),
                 arrowprops=dict(arrowstyle="<-", color="steelblue", lw=2))
    ax1.annotate("", xy=(3, 2.2), xytext=(0, 1.0),
                 arrowprops=dict(arrowstyle="<-", color="steelblue", lw=2))
    ax1.text(0, 0.7, "Reflected particle\n반사된 입자 (bounces back)",
             ha="center", fontsize=10, color="steelblue", fontweight="bold")

    # Particle 2: small pitch angle → penetrates to atmosphere
    ax1.annotate("", xy=(-4.5, B[50]), xytext=(0, 1.0),
                 arrowprops=dict(arrowstyle="->", color="red", lw=2, ls="--"))
    ax1.text(-4.2, 3.5, "Precipitating particle\n강하 입자 → AURORA!",
             fontsize=10, color="red", fontweight="bold")

    ax1.set_xlabel("Distance along field line / 자기장선을 따른 거리")
    ax1.set_ylabel("$|B|$ (normalized) / 자기장 세기 (정규화)")
    ax1.set_title("Magnetic Mirror Effect\n자기 거울 효과")
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Mark mirror points
    ax1.axhline(2.5, color="gray", ls=":", lw=1)
    ax1.text(4, 2.6, "Mirror point\n(거울점)", fontsize=9, color="gray")

    # --- Right: Loss cone ---
    ax2 = axes[1]

    # Loss cone angle depends on mirror ratio
    L_values = np.arange(2, 11)
    # Mirror ratio for dipole: B_mirror/B_eq = sqrt(1 + 3*sin^2(lam)) / (cos^6(lam) * ...)
    # Simplified: at equator of L-shell, B_eq = B0/L^3
    # At surface (r=1): B_surface = B0 * sqrt(1+3*sin^2(lam_foot))
    # For L-shell, footpoint latitude: cos^2(lam_foot) = 1/L
    loss_cone_deg = []
    for L in L_values:
        cos2_lam = 1.0 / L
        sin2_lam = 1 - cos2_lam
        # Mirror ratio
        B_ratio = L**3 * np.sqrt(1 + 3 * sin2_lam) / (1.0)
        alpha_lc = np.degrees(np.arcsin(np.sqrt(1.0 / B_ratio)))
        loss_cone_deg.append(alpha_lc)

    ax2.bar(L_values, loss_cone_deg, color="red", alpha=0.7, edgecolor="darkred")
    for L, alpha in zip(L_values, loss_cone_deg):
        ax2.text(L, alpha + 0.5, f"{alpha:.1f}°", ha="center", fontsize=9)

    ax2.set_xlabel("L-shell value / L-shell 값")
    ax2.set_ylabel("Loss cone half-angle (°) / 손실 원뿔 반각 (°)")
    ax2.set_title("Loss Cone Angle vs. L-shell\n손실 원뿔 각도 vs. L-shell\n"
                  "(Smaller cone = fewer particles reach atmosphere)")
    ax2.grid(True, alpha=0.3, axis="y")

    # Annotate auroral L-shell
    ax2.axvline(5, color="green", ls="--", lw=2, alpha=0.7)
    ax2.text(5.3, max(loss_cone_deg) * 0.9, "Auroral zone\n(L≈5, ~67°)",
             fontsize=10, color="green")

    plt.tight_layout()
    plt.show()

    print("Key insight (핵심 통찰):")
    print("Only particles with pitch angle < loss cone angle reach the atmosphere.")
    print("At L=5 (auroral zone), loss cone ≈ 4° — very narrow!")
    print("This is why aurora form thin arcs, not diffuse glows.")

plot_magnetic_mirror()

## Part 4: Terrella Experiment Simulation / Terrella 실험 시뮬레이션

Birkeland's most brilliant innovation: shoot electrons at a magnetized sphere
in a vacuum chamber and observe where they land. The result: a glowing ring
around the magnetic poles — an **artificial aurora**.

We simulate this by launching many particles toward a dipole and recording
where they impact the sphere surface.

Birkeland의 가장 독창적인 혁신: 진공 챔버에서 자화된 구체에 전자를 쏘고
어디에 도달하는지 관찰. 결과: 자기 극 근처의 빛나는 고리 — **인공 오로라**.

In [ ]:
def terrella_simulation(n_particles=500, q_over_m=30.0):
    """Simulate Birkeland's terrella experiment.

    Launch particles toward a magnetized sphere and record impact locations.

    Args:
        n_particles: Number of particles to launch.
        q_over_m: Charge-to-mass ratio (normalized).

    Returns:
        impact_lats: Latitudes where particles hit the sphere.
        impact_lons: Longitudes where particles hit the sphere.
    """
    np.random.seed(42)
    impact_lats = []
    impact_lons = []

    for _ in range(n_particles):
        # Launch from x = +8 R_E, random y, z offsets (simulating beam spread)
        y0 = np.random.uniform(-3, 3)
        z0 = np.random.uniform(-3, 3)
        pos = np.array([8.0, y0, z0])

        # Velocity toward the terrella (negative x direction) with small spread
        vx = -0.5 + np.random.normal(0, 0.02)
        vy = np.random.normal(0, 0.02)
        vz = np.random.normal(0, 0.02)
        vel = np.array([vx, vy, vz])

        dt = 0.02
        for step in range(15000):
            r = np.linalg.norm(pos)
            if r > 15:
                break
            if r < 1.05:
                # Hit the sphere! Record latitude and longitude
                lat = np.degrees(np.arcsin(pos[2] / r))
                lon = np.degrees(np.arctan2(pos[1], pos[0]))
                impact_lats.append(lat)
                impact_lons.append(lon)
                break

            Bx, By, Bz = dipole_B_cartesian(pos[0], pos[1], pos[2])
            B = np.array([Bx, By, Bz])
            accel = q_over_m * np.cross(vel, B)
            vel = vel + accel * dt
            pos = pos + vel * dt

    return np.array(impact_lats), np.array(impact_lons)


# Run the terrella simulation
print("Running terrella simulation (500 particles)...")
impact_lats, impact_lons = terrella_simulation(n_particles=500)
print(f"Particles that hit the terrella: {len(impact_lats)} / 500")

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Impact locations in lat/lon (Mollweide projection)
ax1 = axes[0]
ax1.scatter(impact_lons, impact_lats, c="lime", s=8, alpha=0.6, edgecolors="none")
ax1.axhline(67, color="red", ls="--", lw=1, alpha=0.7, label="67° (auroral zone)")
ax1.axhline(-67, color="red", ls="--", lw=1, alpha=0.7)
ax1.set_xlabel("Longitude (°) / 경도 (°)")
ax1.set_ylabel("Latitude (°) / 위도 (°)")
ax1.set_title("Terrella: Particle Impact Locations\nTerrella: 입자 충돌 위치")
ax1.set_xlim(-180, 180)
ax1.set_ylim(-90, 90)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_facecolor("black")

# Right: Latitude histogram
ax2 = axes[1]
if len(impact_lats) > 0:
    ax2.hist(impact_lats, bins=36, range=(-90, 90), color="lime",
             edgecolor="darkgreen", alpha=0.8)
    ax2.axvline(67, color="red", ls="--", lw=2, label="67° auroral zone")
    ax2.axvline(-67, color="red", ls="--", lw=2)
ax2.set_xlabel("Impact latitude (°) / 충돌 위도 (°)")
ax2.set_ylabel("Count / 개수")
ax2.set_title("Distribution of Impact Latitudes\n충돌 위도 분포\n"
              "(Particles concentrate near magnetic poles!)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

if len(impact_lats) > 0:
    print(f"\nMean |latitude| of impacts: {np.mean(np.abs(impact_lats)):.1f}°")
    print(f"Median |latitude|: {np.median(np.abs(impact_lats)):.1f}°")
    print("→ Particles preferentially hit near the POLES, forming an AURORAL ZONE!")
    print("   This is exactly what Birkeland observed in his terrella experiments.")

## Part 5: Birkeland's Current System / Birkeland 전류 체계

Birkeland proposed that currents flow **along magnetic field lines** from
space into the polar atmosphere (and back). This was his most controversial
claim, rejected by Chapman for decades, and confirmed by satellite in 1966.

Birkeland는 전류가 자기장선을 따라 우주 공간에서 극지방 대기로 **흐른다고** 제안했습니다.
이것은 그의 가장 논쟁적인 주장이었고, Chapman에 의해 수십 년간 거부된 후
1966년 위성으로 확인되었습니다.

```
        Solar wind →→→
                     ╭──────╮
                    ╱  ↓↓↓↓  ╲    ← Birkeland currents (field-aligned)
                   │  ○○○○○○  │   ← Auroral electrojet (horizontal)
                   │  EARTH   │
                   │  ○○○○○○  │
                    ╲  ↑↑↑↑  ╱    ← Return currents
                     ╰──────╯
```

In [ ]:
def plot_birkeland_currents():
    """Visualize Birkeland's field-aligned current system."""
    fig, ax = plt.subplots(figsize=(14, 10))

    # Draw Earth (polar view from above north pole)
    earth = Circle((0, 0), 1.0, color="steelblue", alpha=0.6, zorder=5)
    ax.add_patch(earth)
    ax.text(0, 0, "North Pole\n(top view)\n북극 (상면도)",
            ha="center", va="center", fontsize=9, color="white",
            fontweight="bold", zorder=6)

    # Auroral oval (approximately at 67° = 23° from pole)
    auroral_r = np.sin(np.radians(23))  # radius in polar projection
    auroral_oval = Circle((0, 0), auroral_r, fill=False, color="lime",
                          lw=3, ls="--", zorder=4, label="Auroral oval / 오로라 타원")
    ax.add_patch(auroral_oval)

    # Region 1 Birkeland currents (higher latitude, connected to magnetopause)
    r1_r = auroral_r - 0.03
    theta_r1 = np.linspace(0, 2 * np.pi, 24)
    for i, th in enumerate(theta_r1):
        x = r1_r * np.cos(th)
        y = r1_r * np.sin(th)
        # Alternating in/out (dawn/dusk asymmetry)
        if np.cos(th) > 0:  # dusk side: current INTO ionosphere
            ax.plot(x, y, "rv", ms=8, zorder=7)  # down arrow = into page
        else:  # dawn side: current OUT of ionosphere
            ax.plot(x, y, "r^", ms=8, zorder=7)  # up arrow = out of page

    # Region 2 Birkeland currents (lower latitude, connected to ring current)
    r2_r = auroral_r + 0.05
    theta_r2 = np.linspace(0, 2 * np.pi, 24)
    for i, th in enumerate(theta_r2):
        x = r2_r * np.cos(th)
        y = r2_r * np.sin(th)
        if np.cos(th) > 0:  # opposite sense to Region 1
            ax.plot(x, y, "b^", ms=6, zorder=7)
        else:
            ax.plot(x, y, "bv", ms=6, zorder=7)

    # Pedersen currents (horizontal, connecting R1 and R2)
    for th in np.linspace(0, 2 * np.pi, 12):
        x1 = r1_r * np.cos(th)
        y1 = r1_r * np.sin(th)
        x2 = r2_r * np.cos(th)
        y2 = r2_r * np.sin(th)
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="->", color="orange", lw=1.5, alpha=0.5))

    # Labels
    ax.annotate("Dusk / 해질녘", (0.55, 0), fontsize=12, ha="center", color="gray")
    ax.annotate("Dawn / 새벽", (-0.55, 0), fontsize=12, ha="center", color="gray")
    ax.annotate("Noon / 정오\n(Sun side / 태양 방향)", (0, 0.55), fontsize=11,
                ha="center", color="gold")
    ax.annotate("Midnight / 자정", (0, -0.55), fontsize=11, ha="center", color="gray")

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Circle((0, 0), 0.01, color="lime", label="Auroral oval / 오로라 타원"),
        Line2D([0], [0], marker="v", color="w", markerfacecolor="red", ms=10,
               label="Region 1: current INTO ionosphere / 전리층으로 유입"),
        Line2D([0], [0], marker="^", color="w", markerfacecolor="red", ms=10,
               label="Region 1: current OUT of ionosphere / 전리층에서 유출"),
        Line2D([0], [0], marker="^", color="w", markerfacecolor="blue", ms=8,
               label="Region 2: return currents / 귀환 전류"),
        Line2D([0], [0], color="orange", lw=2,
               label="Pedersen currents (horizontal) / Pedersen 전류 (수평)"),
    ]
    ax.legend(handles=legend_elements, loc="upper right", fontsize=9,
              bbox_to_anchor=(1.35, 1.0))

    ax.set_xlim(-0.8, 0.8)
    ax.set_ylim(-0.7, 0.7)
    ax.set_aspect("equal")
    ax.set_title("Birkeland Current System (Polar View)\n"
                 "Birkeland 전류 체계 (극지방 상면도)\n"
                 "▼=current into page, ▲=current out of page", fontsize=13)
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    print("Birkeland's key insight: currents flow ALONG field lines")
    print("from space into the polar ionosphere (and back).")
    print("This was confirmed by satellite (Zmuda et al., 1966).")
    print(f"\nTotal Birkeland current magnitude: ~10⁶ A (millions of amperes!)")

plot_birkeland_currents()

## Summary / 요약

| Part | What we simulated / 시뮬레이션한 것 | Birkeland's concept / Birkeland의 개념 |
|------|----------------------------------|--------------------------------------|
| 1 | Dipole field lines + field strength | Earth's magnetic geometry concentrates particles at poles / 지구 자기장이 입자를 극지방에 집중 |
| 2 | Particle spiraling in dipole field | Cathode rays follow field lines to aurora / 전자가 자기장선을 따라 오로라 지역으로 |
| 3 | Magnetic mirror + loss cone | Only specific particles reach atmosphere / 특정 입자만 대기에 도달 |
| 4 | Terrella experiment | Artificial aurora on magnetized sphere / 자화 구체 위의 인공 오로라 |
| 5 | Birkeland current system | Field-aligned currents: space ↔ atmosphere / 자기장 정렬 전류: 우주 공간 ↔ 대기 |

**Next paper / 다음 논문**: #2 Chapman & Ferraro (1931) — "A New Theory of Magnetic Storms."
The opposing theory that dominated for decades, proposing that magnetic storms
are caused by electrically neutral plasma streams compressing Earth's field.
Ironically, the eventual resolution combined elements of both Birkeland AND Chapman.

다음 논문: #2 Chapman & Ferraro (1931) — "자기폭풍의 새 이론."
수십 년간 지배했던 반대 이론. 전기적으로 중성인 플라즈마 흐름이 지구 자기장을
압축하여 자기폭풍을 일으킨다고 제안. 아이러니하게도 최종 해답은 Birkeland와
Chapman 양쪽의 요소를 결합한 것이었습니다.